"# 17 — GCD Backbone → NASA GLOBE (Proportional Patch Training + Cloud Filter)\n",
    "\n",
    "Same GCD backbone and 3-phase unfreezing as notebook 16, with two changes:\n",
    "\n",
    "1. **Proportional patches** — each image is split into overlapping crops whose\n",
    "   side length is 50% of the shorter image dimension (so a 1080×810 image gets\n",
    "   ~405px crops, a 4032×3024 gets ~1512px crops). Every crop is resized to\n",
    "   224×224 and treated as an independent training example with its parent label.\n",
    "\n",
    "2. **Patch cloud filter** — each patch is scored by the trained ResNet18\n",
    "   binary classifier (head2). Patches with cloud confidence < 0.5 are discarded\n",
    "   before training, removing ground/wall/tree fragments that crept in at the\n",
    "   edges of wide-angle images.\n",
    "\n",
    "Splits are performed at **image level** to prevent patches from the same\n",
    "parent appearing in both train and val.\n",
    "\n",
    "**Patch parameters** (tune in the data cell):\n",
    "- `CROP_FRACTION = 0.5` — crop side as fraction of the shorter image side\n",
    "- `OVERLAP = 0.25` — fractional overlap between adjacent patches\n",
    "- `CLOUD_THRESHOLD = 0.5` — minimum head2 cloud confidence to keep a patch\n",
    "\n",
    "**Data:** `extract_filtered-thr99-sam3000` (pre-filtered, 10 classes)."

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

IMAGES_ROOT   = Path('../resources/cloud-images/NASA_GLOBE_CD/extract_filtered-thr99-sam3000')
CROP_FRACTION = 0.5    # crop side = 50% of the shorter image dimension
OVERLAP       = 0.25   # 25% overlap between adjacent patches
OUTPUT_SIZE   = 224

def index_images(images_root):
    """Scan class subdirectories → flat (path, label) lists."""
    paths, labels = [], []
    for cls_dir in sorted(images_root.iterdir()):
        if not cls_dir.is_dir():
            continue
        for p in cls_dir.iterdir():
            if p.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                paths.append(str(p))
                labels.append(cls_dir.name)
    return paths, np.array(labels)

paths, labels = index_images(IMAGES_ROOT)

counts = Counter(labels)
for cls in sorted(counts):
    print(f'  {cls:6s}  {counts[cls]:,}')
print(f'  {"TOTAL":6s}  {sum(counts.values()):,}')

  Ac      3,000
  As      3,000
  Cb      3,000
  Cc      3,000
  Ci      3,000
  Cs      3,000
  Cu      3,000
  Ns      3,000
  Sc      3,000
  St      3,000
  TOTAL   30,000


In [8]:
import torch

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f'device: {device}')

device: mps


In [9]:
import torchvision.transforms.v2 as T
from torchvision.transforms.v2 import InterpolationMode

IMG_SIZE = (OUTPUT_SIZE, OUTPUT_SIZE)
MAX_FRAC = 0.14

border_translation = T.RandomAffine(
    degrees=0,
    translate=(MAX_FRAC, MAX_FRAC),
    interpolation=InterpolationMode.BILINEAR,
    fill=0
)

# T.Lambda is fine locally with num_workers=0
wrap_translation = T.Lambda(lambda x: torch.roll(
    x,
    shifts=(
        int(torch.randint(-int(MAX_FRAC * x.shape[-2]), int(MAX_FRAC * x.shape[-2]) + 1, (1,)).item()),
        int(torch.randint(-int(MAX_FRAC * x.shape[-1]), int(MAX_FRAC * x.shape[-1]) + 1, (1,)).item()),
    ),
    dims=(-2, -1),
))

stacked = T.Compose([
    T.RandomChoice([wrap_translation, border_translation]),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.15),
    T.RandomAffine(
        degrees=20, scale=(0.90, 1.10),
        interpolation=InterpolationMode.BILINEAR, fill=0
    ),
])

one_of = T.RandomChoice([
    wrap_translation,
    border_translation,
    T.RandomChoice([T.RandomHorizontalFlip(p=1.0), T.RandomVerticalFlip(p=1.0)]),
    T.RandomRotation(degrees=20, interpolation=InterpolationMode.BILINEAR, fill=0),
])

normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

# Patches are already 224×224 — no Resize or RandomResizedCrop needed.
# Augmentation is applied after the patch is extracted.
train_transforms = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.RandomChoice([stacked, one_of]),
    normalize,
])

eval_transforms = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    normalize,
])

In [10]:
from sklearn.preprocessing import LabelEncoder

def encode_labels(labels):
    enc = LabelEncoder()
    encoded = enc.fit_transform(labels)
    return encoded, enc.classes_

encoded_labels, class_names = encode_labels(labels)
n_classes = len(class_names)
print(f'Classes ({n_classes}): {class_names}')

Classes (10): ['Ac' 'As' 'Cb' 'Cc' 'Ci' 'Cs' 'Cu' 'Ns' 'Sc' 'St']


In [11]:
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedShuffleSplit
from PIL import Image


class PatchDataset(Dataset):
    """
    Expands each image into proportional 224×224 patches.

    Crop boxes are computed once at init (fast — only reads image headers).
    Actual pixel data is loaded lazily in __getitem__.
    Each patch inherits its parent image's label.
    """
    def __init__(self, paths, encoded_labels,
                 crop_fraction=0.5, overlap=0.25, output_size=224,
                 transform=None):
        self.output_size = output_size
        self.transform   = transform
        self.samples     = []  # (path, label, (x0, y0, x1, y1))

        for path, label in zip(paths, encoded_labels):
            try:
                W, H = Image.open(path).size
            except Exception:
                continue
            crop_size = max(output_size, int(min(W, H) * crop_fraction))
            stride    = int(crop_size * (1 - overlap))
            for y in range(0, H - crop_size + 1, stride):
                for x in range(0, W - crop_size + 1, stride):
                    self.samples.append((path, label, (x, y, x + crop_size, y + crop_size)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, box = self.samples[idx]
        img  = Image.open(path).convert('RGB')
        crop = img.crop(box).resize((self.output_size, self.output_size), Image.BILINEAR)
        if self.transform:
            crop = self.transform(crop)
        return crop, label


# ── Split at IMAGE level (not patch level) to prevent leakage ────────────────
paths_arr  = np.array(paths)
labels_arr = np.array(encoded_labels)

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
trainval_idx, test_idx = next(sss1.split(paths_arr, labels_arr))

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.1 / 0.8, random_state=42)
train_rel_idx, val_rel_idx = next(
    sss2.split(paths_arr[trainval_idx], labels_arr[trainval_idx]))

train_paths = paths_arr[trainval_idx[train_rel_idx]]
train_lbls  = labels_arr[trainval_idx[train_rel_idx]]
val_paths   = paths_arr[trainval_idx[val_rel_idx]]
val_lbls    = labels_arr[trainval_idx[val_rel_idx]]
test_paths  = paths_arr[test_idx]
test_lbls   = labels_arr[test_idx]

print('Building patch datasets (reading image headers)...')
train_set = PatchDataset(train_paths, train_lbls,
                         CROP_FRACTION, OVERLAP, OUTPUT_SIZE, train_transforms)
valid_set = PatchDataset(val_paths,   val_lbls,
                         CROP_FRACTION, OVERLAP, OUTPUT_SIZE, eval_transforms)
test_set  = PatchDataset(test_paths,  test_lbls,
                         CROP_FRACTION, OVERLAP, OUTPUT_SIZE, eval_transforms)

print(f'Images  — train: {len(train_paths):,}  val: {len(val_paths):,}  test: {len(test_paths):,}')
print(f'Patches — train: {len(train_set):,}  val: {len(valid_set):,}  test: {len(test_set):,}')
avg_patches = len(train_set) / len(train_paths)
print(f'Avg patches per image: {avg_patches:.1f}')

Building patch datasets (reading image headers)...
Images  — train: 21,000  val: 3,000  test: 6,000
Patches — train: 138,652  val: 19,812  test: 39,592
Avg patches per image: 6.6


In [12]:
import torch.nn as nn
import torchvision
import torchvision.transforms as T_filter
import torch.nn.functional as F_filter
from collections import Counter

CLOUD_THRESHOLD  = 0.5    # keep patches with head2 cloud confidence >= this
FILTER_BATCH     = 128


class MultiHeadResNet18(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(
            weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.head1 = base.fc
        self.head2 = nn.Linear(512, num_classes)
        for p in self.backbone.parameters(): p.requires_grad = False
        for p in self.head1.parameters():    p.requires_grad = False

    def forward(self, x):
        feat = self.backbone(x).flatten(1)
        return self.head1(feat), self.head2(feat)


filter_ckpt    = torch.load('../models/multihead_resnet18_cloud_sky.pth',
                            map_location='cpu', weights_only=False)
filter_classes = filter_ckpt['classes']          # ['clear_sky', 'cloud']
cloud_idx      = filter_classes.index('cloud')
filter_model   = MultiHeadResNet18(num_classes=len(filter_classes))
filter_model.load_state_dict(filter_ckpt['model_state_dict'])
filter_model.eval().to(device)

filter_transform = T_filter.Compose([
    T_filter.ToTensor(),
    T_filter.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def filter_patch_samples(samples, threshold=CLOUD_THRESHOLD, batch_size=FILTER_BATCH):
    """
    Keep only patches where the ResNet18 cloud classifier gives
    head2 cloud confidence >= threshold.

    Patches are grouped by parent image path so each image file is
    opened exactly once.  Inference runs in batches of batch_size.
    """
    kept = []
    tensor_buf, sample_buf = [], []
    current_path, current_img = None, None

    def flush():
        if not tensor_buf:
            return
        inp = torch.stack(tensor_buf).to(device)
        with torch.no_grad():
            _, out2 = filter_model(inp)
            probs = F_filter.softmax(out2, dim=1).cpu().numpy()
        for samp, prob in zip(sample_buf, probs):
            if prob[cloud_idx] >= threshold:
                kept.append(samp)
        tensor_buf.clear()
        sample_buf.clear()

    for sample in samples:
        path, label, box = sample
        if path != current_path:
            try:
                current_img  = Image.open(path).convert('RGB')
                current_path = path
            except Exception:
                current_img  = None
                current_path = path
        if current_img is None:
            continue
        try:
            crop = current_img.crop(box).resize((OUTPUT_SIZE, OUTPUT_SIZE), Image.BILINEAR)
            tensor_buf.append(filter_transform(crop))
            sample_buf.append(sample)
        except Exception:
            pass
        if len(tensor_buf) >= batch_size:
            flush()

    flush()
    return kept


def summarise_filter(before, after, split_name):
    n_before = len(before)
    n_after  = len(after)
    kept_pct = 100 * n_after / n_before if n_before else 0
    print(f'  {split_name:5s}  {n_before:>8,} → {n_after:>8,}  ({kept_pct:.1f}% kept)')
    before_cls = Counter(s[1] for s in before)
    after_cls  = Counter(s[1] for s in after)
    for cls_idx in sorted(before_cls):
        b, a = before_cls[cls_idx], after_cls.get(cls_idx, 0)
        pct  = 100 * a / b if b else 0
        print(f'         class {class_names[cls_idx]:4s}  {b:>6,} → {a:>6,}  ({pct:.1f}%)')


print(f'Patch cloud filter  threshold={CLOUD_THRESHOLD}\n')

for ds, name in [(train_set, 'train'), (valid_set, 'val'), (test_set, 'test')]:
    before = ds.samples[:]
    ds.samples = filter_patch_samples(ds.samples)
    summarise_filter(before, ds.samples, name)
    print()

Patch cloud filter  threshold=0.5

  train   138,652 →  129,041  (93.1% kept)
         class Ac    13,802 → 12,777  (92.6%)
         class As    13,780 → 13,088  (95.0%)
         class Cb    13,686 → 12,806  (93.6%)
         class Cc    14,226 → 13,194  (92.7%)
         class Ci    13,926 → 12,313  (88.4%)
         class Cs    14,100 → 13,099  (92.9%)
         class Cu    14,172 → 12,654  (89.3%)
         class Ns    13,628 → 12,999  (95.4%)
         class Sc    13,650 → 13,081  (95.8%)
         class St    13,682 → 13,030  (95.2%)

  val      19,812 →   18,431  (93.0% kept)
         class Ac     1,970 →  1,835  (93.1%)
         class As     1,956 →  1,859  (95.0%)
         class Cb     1,954 →  1,859  (95.1%)
         class Cc     2,028 →  1,872  (92.3%)
         class Ci     1,964 →  1,729  (88.0%)
         class Cs     1,998 →  1,872  (93.7%)
         class Cu     2,030 →  1,814  (89.4%)
         class Ns     1,930 →  1,818  (94.2%)
         class Sc     1,956 →  1,858  (95.0%)
    

In [19]:
import random as _random

MAX_PATCHES_PER_CLASS = 3_000
_rng = _random.Random(42)

by_class = {}
for sample in train_set.samples:
    by_class.setdefault(sample[1], []).append(sample)

capped = []
print(f'Capping training patches to {MAX_PATCHES_PER_CLASS:,}/class (random, seed=42)\n')
for cls_idx in sorted(by_class):
    pool   = by_class[cls_idx]
    chosen = _rng.sample(pool, min(MAX_PATCHES_PER_CLASS, len(pool)))
    capped.extend(chosen)
    print(f'  {class_names[cls_idx]:4s}  {len(pool):>7,} → {len(chosen):>5,}')

train_set.samples = capped
print(f'\n  Total  {sum(len(v) for v in by_class.values()):>7,} → {len(train_set.samples):>5,}')

Capping training patches to 3,000/class (random, seed=42)

  Ac     12,777 → 3,000
  As     13,088 → 3,000
  Cb     12,806 → 3,000
  Cc     13,194 → 3,000
  Ci     12,313 → 3,000
  Cs     13,099 → 3,000
  Cu     12,654 → 3,000
  Ns     12,999 → 3,000
  Sc     13,081 → 3,000
  St     13,030 → 3,000

  Total  129,041 → 30,000


In [26]:
from torch.utils.data import DataLoader, WeightedRandomSampler

train_labels_list = [s[1] for s in train_set.samples]
class_counts  = np.bincount(train_labels_list)
class_weights = 1.0 / class_counts
sample_weights = torch.tensor([class_weights[l] for l in train_labels_list], dtype=torch.float)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# num_workers=0 required locally — T.Lambda in transforms cannot be pickled
train_loader = DataLoader(train_set, batch_size=64, sampler=sampler,  num_workers=0)
valid_loader = DataLoader(valid_set, batch_size=64, shuffle=False,    num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False,    num_workers=0)

In [27]:
import torchvision
import torch.nn as nn

GCD_N_CLASSES = 7
GCD_CKPT_PATH = 'best_cloudensenet_colab_15_1.pt'


class TopBlock(nn.Module):
    """
    Custom classification head from CloudDenseNet (Li et al., Sensors 2023).

    BN → Dropout → Linear(1024, hidden) → ReLU → BN → Dropout → Linear(hidden, n_classes)
    Weights initialised with LeCun uniform distribution.
    """
    def __init__(self, in_features: int, n_classes: int,
                 hidden_dim: int = 512, dropout: float = 0.5):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(p=dropout),
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(p=dropout / 2),
            nn.Linear(hidden_dim, n_classes),
        )
        self._lecun_init()

    def _lecun_init(self):
        for m in self.block:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.block(x)


# Step 1 — DenseNet121 + 7-class head to match checkpoint shape
weights = torchvision.models.DenseNet121_Weights.IMAGENET1K_V1
model = torchvision.models.densenet121(weights=weights).to(device)
model.classifier = TopBlock(in_features=1024, n_classes=GCD_N_CLASSES).to(device)

# Step 2 — load GCD backbone weights
model.load_state_dict(torch.load(GCD_CKPT_PATH, map_location=device, weights_only=True))
print('GCD checkpoint loaded')

# Step 3 — freeze backbone, replace head with fresh 10-class TopBlock
for param in model.parameters():
    param.requires_grad = False

model.classifier = TopBlock(in_features=1024, n_classes=n_classes).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Head replaced: GCD {GCD_N_CLASSES}-class → NASA GLOBE {n_classes}-class')
print(f'Classes ({n_classes}): {class_names}')
print(f'Trainable params: {trainable:,} / {total:,}')

GCD checkpoint loaded
Head replaced: GCD 7-class → NASA GLOBE 10-class
Classes (10): ['Ac' 'As' 'Cb' 'Cc' 'Ci' 'Cs' 'Cu' 'Ns' 'Sc' 'St']
Trainable params: 533,002 / 7,486,858


In [28]:
import torchmetrics


def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            metric.update(model(X_batch), y_batch)
    return metric.compute()


accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=n_classes).to(device)

In [29]:
import torch.nn.functional as F


class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.exp(-ce)
        loss = ((1.0 - p_t) ** self.gamma) * ce
        return loss.mean() if self.reduction == 'mean' else loss.sum()


focal_loss = FocalLoss(gamma=2.0).to(device)
print('FocalLoss ready (gamma=2.0)')

FocalLoss ready (gamma=2.0)


In [ ]:
import time
from tqdm.auto import tqdm


def train_phase(model, optimizer, loss_fn, metric,
                train_loader, valid_loader,
                n_epochs, patience, checkpoint_path, phase_label=''):
    history    = {'train_losses': [], 'train_metrics': [], 'valid_metrics': []}
    best_val   = 0.0
    no_improve = 0

    for epoch in range(n_epochs):
        t0 = time.time()

        model.eval()
        model.classifier.train()
        for module in model.modules():
            if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
                if any(p.requires_grad for p in module.parameters()):
                    module.train()

        total_loss = 0.0
        metric.reset()

        for X_batch, y_batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{n_epochs}',
                                     leave=False, unit='batch'):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss   = loss_fn(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            metric.update(y_pred, y_batch)

        train_loss = total_loss / len(train_loader)
        train_acc  = metric.compute().item()
        val_acc    = evaluate_tm(model, valid_loader, metric).item()
        elapsed    = (time.time() - t0) / 60.0

        history['train_losses'].append(train_loss)
        history['train_metrics'].append(train_acc)
        history['valid_metrics'].append(val_acc)

        star = ' *' if val_acc > best_val else ''
        print(f'[{phase_label}] Epoch {epoch+1}/{n_epochs} | '
              f'loss: {train_loss:.4f} | '
              f'train: {train_acc:.4f} | '
              f'val: {val_acc:.4f} | '
              f'{elapsed:.2f} min{star}')

        if val_acc > best_val:
            best_val   = val_acc
            no_improve = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at epoch {epoch+1} (best val: {best_val:.4f})')
                break

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return history, best_val

In [ ]:
all_history = []
CKPT1 = 'best_patch17_phase1.pt'
CKPT2 = 'best_patch17_phase2.pt'
CKPT3 = 'best_patch17_phase3.pt'

# ── Phase 1 — Head only, lr=1e-4, 20 epochs ──────────────────────────────────
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-4, weight_decay=1e-4)
h1, phase1_best_val = train_phase(
    model, opt, focal_loss, accuracy,
    train_loader, valid_loader,
    n_epochs=20, patience=10,
    checkpoint_path=CKPT1,
    phase_label='Phase 1 (head, 1e-4)')
all_history.append(h1)
print(f'Phase 1 best val: {phase1_best_val:.4f}')

# ── Phase 2 — Head + denseblock4 + transition3, 30 epochs ───────────────────
model.load_state_dict(torch.load(CKPT1, weights_only=True))

for name, param in model.named_parameters():
    param.requires_grad = (
        name.startswith('classifier') or
        name.startswith('features.denseblock4') or
        name.startswith('features.transition3')
    )

head_params   = list(model.classifier.parameters())
block4_params = [p for n, p in model.named_parameters()
                 if p.requires_grad and not n.startswith('classifier')]

opt = torch.optim.AdamW([
    {'params': head_params,   'lr': 1e-4},
    {'params': block4_params, 'lr': 1e-5},
], weight_decay=1e-4)
h2, phase2_best_val = train_phase(
    model, opt, focal_loss, accuracy,
    train_loader, valid_loader,
    n_epochs=30, patience=10,
    checkpoint_path=CKPT2,
    phase_label='Phase 2 (block4+tr3, 1e-5)')
all_history.append(h2)
print(f'Phase 2 best val: {phase2_best_val:.4f}')

# ── Phase 3 — conditional: also unfreeze denseblock3 + transition2 ───────────
FINAL_CKPT  = CKPT2
improvement = phase2_best_val - phase1_best_val

if improvement > 0.01:
    print(f'Phase 2 gain {improvement:.4f} > 1pp — running Phase 3')
    model.load_state_dict(torch.load(CKPT2, weights_only=True))

    for name, param in model.named_parameters():
        param.requires_grad = (
            name.startswith('classifier') or
            name.startswith('features.denseblock4') or
            name.startswith('features.transition3') or
            name.startswith('features.denseblock3') or
            name.startswith('features.transition2')
        )

    head_params = list(model.classifier.parameters())
    block4_tr3  = [p for n, p in model.named_parameters()
                   if p.requires_grad and not n.startswith('classifier') and
                   (n.startswith('features.denseblock4') or n.startswith('features.transition3'))]
    block3_tr2  = [p for n, p in model.named_parameters()
                   if p.requires_grad and not n.startswith('classifier') and
                   (n.startswith('features.denseblock3') or n.startswith('features.transition2'))]

    opt = torch.optim.AdamW([
        {'params': head_params, 'lr': 1e-4},
        {'params': block4_tr3,  'lr': 1e-5},
        {'params': block3_tr2,  'lr': 5e-6},
    ], weight_decay=1e-4)
    h3, phase3_best_val = train_phase(
        model, opt, focal_loss, accuracy,
        train_loader, valid_loader,
        n_epochs=20, patience=8,
        checkpoint_path=CKPT3,
        phase_label='Phase 3 (block3+tr2, 5e-6)')
    all_history.append(h3)
    FINAL_CKPT = CKPT3
    print(f'Phase 3 best val: {phase3_best_val:.4f}')
else:
    print(f'Phase 2 gain {improvement:.4f} ≤ 1pp — skipping Phase 3')

print(f'\nFinal checkpoint: {FINAL_CKPT}')
model.load_state_dict(torch.load(FINAL_CKPT, weights_only=True))

Phase 1 (head, 1e-4):   0%|          | 0/20 [00:00<?, ?epoch/s]

Phase 1 (head, 1e-4):   5%|▌         | 1/20 [10:02<3:10:40, 602.14s/epoch]

[Phase 1 (head, 1e-4)] Epoch 1/20 | loss: 1.9750 | train: 0.1920 | val: 0.2427 | 10.03 min *


In [ ]:
test_acc = evaluate_tm(model, test_loader, accuracy)
print(f'Final test accuracy: {test_acc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['tab:blue', 'tab:green', 'tab:red']
phase_labels = [
    'Phase 1 (head only, 1e-4)',
    'Phase 2 (block4+tr3, 1e-5)',
    'Phase 3 (block3+tr2, 5e-6)',
]

offset = 0
for h, label, color in zip(all_history, phase_labels, colors):
    xs = list(range(offset, offset + len(h['train_losses'])))
    axes[0].plot(xs, h['train_losses'], color=color, label=label)
    axes[1].plot(xs, h['train_metrics'], color=color, linestyle='--', alpha=0.5)
    axes[1].plot(xs, h['valid_metrics'],  color=color, label=label)
    for ax in axes:
        ax.axvline(x=offset, color='grey', linewidth=0.5, linestyle=':')
    offset += len(h['train_losses'])

axes[0].set_title('Focal Loss — GCD backbone → NASA GLOBE (patch training)')
axes[0].set_xlabel('Epoch (cumulative)')
axes[0].set_ylabel('Focal Loss')
axes[0].legend(fontsize=8)

axes[1].set_title('Accuracy — dashed=train, solid=val')
axes[1].set_xlabel('Epoch (cumulative)')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds = torch.argmax(model(X_batch), dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y_batch.cpu().numpy())

cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix on Test Set (patch-level predictions)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()